# Exploring Stock Data using brapi-dev api

## Load ingested data into spark tables

In [1]:
from pyspark.sql import SparkSession
from spark.loader import load_all_tables

spark = (
    SparkSession.builder
    .appName("BrapiDataLake")
    .master("local[*]")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 12:09:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
load_all_tables(spark)

26/06/06 12:09:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
spark.catalog.listTables()

[Table(name='activetickers', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='balancesheethistoryquarterly', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='cashflowhistoryquarterly', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='defaultkeystatistics', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='defaultkeystatisticshistoryquarterly', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='financialdatahistoryquarterly', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='incomestatementhistoryquarterly', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='valueaddedhistoryquarterly', catalog=None, namespace=[], description=None, tab

## Explore

In [4]:
spark.table("activetickers").show(5)

+------+-----+--------------------+----------------+--------------------+-------------------+-----+-------+-----+--------+
|change|close|                logo|      market_cap|                name|             sector|stock|subType| type|  volume|
+------+-----+--------------------+----------------+--------------------+-------------------+-----+-------+-----+--------+
| -1.84|19.17|https://icons.bra...|1.11688488341E11|     BCO BRASIL S.A.|            Finance|BBAS3|  stock|stock|51043500|
|  0.28| 3.59|https://icons.bra...| 1.4172060743E10|          COSAN S.A.|          Utilities|CSAN3|  stock|stock|40491100|
|  0.28|38.83|https://icons.bra...|4.32650440829E11|ITAU UNIBANCO HOL...|            Finance|ITUB4|  stock|stock|34701800|
| -0.87|40.89|https://icons.bra...|5.61060286488E11|PETROLEO BRASILEI...|    Energy Minerals|PETR4|  stock|stock|34533700|
|-10.18|  6.0|https://icons.bra...|   8.808811888E9|CIA SIDERURGICA N...|Non-Energy Minerals|CSNA3|  stock|stock|34491000|
+------+-----+--

In [5]:
df = spark.table("balancesheethistoryquarterly")
df.select(*[column for column in df.columns][:12]).show(5)

+-----+---------+----------+-----------+--------------------+--------------+-----------+------------------+------------------+-------------------+----------------------+-----------+
|stock|     type|   endDate|       cash|shortTermInvestments|netReceivables|  inventory|otherCurrentAssets|totalCurrentAssets|longTermInvestments|propertyPlantEquipment|otherAssets|
+-----+---------+----------+-----------+--------------------+--------------+-----------+------------------+------------------+-------------------+----------------------+-----------+
|PETR4|quarterly|2026-03-31|34294000000|         13306000000|   22240000000|48556000000|       11371000000|      140533000000|         3091000000|          943869000000|       NULL|
|PETR4|quarterly|2025-12-31|35608000000|         15000001000|   25461000000|45173000000|        7637000000|      140026000000|         3024000000|          924624000000|       NULL|
|PETR4|quarterly|2025-09-30|47675000000|         14326000000|   21891000000|46272000000|  

In [6]:
df = spark.table("cashflowhistoryquarterly")
df.select(*[column for column in df.columns][:9]).show(5)

+-----+---------+----------+-----------------+--------------------+--------------------+-------------------------+-----------------------------+------------------------+
|stock|     type|   endDate|operatingCashFlow|incomeFromOperations|netIncomeBeforeTaxes|adjustmentsToProfitOrLoss|changesInAssetsAndLiabilities|otherOperatingActivities|
+-----+---------+----------+-----------------+--------------------+--------------------+-------------------------+-----------------------------+------------------------+
|PETR4|quarterly|2026-03-31|      43975000000|         63286000000|                NULL|                     NULL|                  -9826000000|             -9485000000|
|PETR4|quarterly|2025-12-31|      54916000000|         60572000000|                NULL|                     NULL|                         NULL|                    NULL|
|PETR4|quarterly|2025-09-30|      53655000000|         68532000000|                NULL|                     NULL|                 -10099000000|      

In [7]:
df = spark.table("defaultkeystatistics")
df.select(*[column for column in df.columns][:10]).show(5)

+------------------------------------+------+
|defaultKeyStatisticsHistoryQuarterly|symbol|
+------------------------------------+------+
|                [{34.540943, 0.07...| PETR4|
|                [{14.525704, 0.06...| MGLU3|
|                [{43.074818, 0.07...| VALE3|
|                [{19.01764, 0.09,...| ITUB4|
+------------------------------------+------+



In [8]:
df = spark.table("defaultkeystatisticshistoryquarterly")
df.select(*[column for column in df.columns][:13]).show(5)

+-----+---------+----------+------------------+---------------+---------+-------------+-----------------+---------+-----------+-----------------------+-----------------+-----------+
|stock|     type|   endDate|             price|enterpriseValue|forwardPE|profitMargins|sharesOutstanding|bookValue|priceToBook|earningsQuarterlyGrowth|netIncomeToCommon|trailingEps|
+-----+---------+----------+------------------+---------------+---------+-------------+-----------------+---------+-----------+-----------------------+-----------------+-----------+
|PETR4|quarterly|2026-03-31|  41.2499961554164|  1161037200000|     NULL|   0.21689811|      12888733000|34.540943|  1.1942348|                   NULL|     107583000000|   8.347058|
|PETR4|quarterly|2025-12-31|  45.4700018121712|  1210129700000|     NULL|   0.22229971|      12888733000|32.399384|  1.4034218|                   NULL|     110605000000|   8.581526|
|PETR4|quarterly|2025-09-30|30.311694991149004|   997604300000|     NULL|   0.15869495|   

In [9]:
df = spark.table("financialdatahistoryquarterly")
df.select(*[column for column in df.columns][:14]).show(5)

+-----+---------+----------+------------+-----------+-----------------+------------+------------+----------+------------+------------+------------+---------------+--------------+
|stock|     type|   endDate|currentPrice|  totalCash|totalCashPerShare|      ebitda|   totalDebt|quickRatio|currentRatio|totalRevenue|debtToEquity|revenuePerShare|returnOnAssets|
+-----+---------+----------+------------+-----------+-----------------+------------+------------+----------+------------+------------+------------+---------------+--------------+
|PETR4|quarterly|2026-03-31|        NULL|47600000000|        3.6931481|230884000000|676977000000|0.48622373|  0.74290836|498091000000|   1.5206507|           NULL|    0.08670072|
|PETR4|quarterly|2025-12-31|        NULL|50608000000|        3.9265304|230015990000|674687000000|0.47816685|  0.70589006|497549000000|   1.6156801|           NULL|    0.09040869|
|PETR4|quarterly|2025-09-30|        NULL|62001000000|        4.8104806|210112000000|668926000000| 0.56528

In [10]:
df = spark.table("incomestatementhistoryquarterly")
df.select(*[column for column in df.columns][:11]).show(5)

+-----+---------+----------+------------+-------------+-----------+-------------------+----------------------------+------------+----------------------+----------------------+
|stock|     type|   endDate|totalRevenue|costOfRevenue|grossProfit|researchDevelopment|sellingGeneralAdministrative|nonRecurring|otherOperatingExpenses|totalOperatingExpenses|
+-----+---------+----------+------------+-------------+-----------+-------------------+----------------------------+------------+----------------------+----------------------+
|PETR4|quarterly|2026-03-31|123686000000| -64084000000|59602000000|               NULL|                 -2517000000|        NULL|           -7899000000|                  NULL|
|PETR4|quarterly|2025-12-31|127371000000| -68878000000|58493000000|               NULL|                 -2854000100|        NULL|          -18089000000|                  NULL|
|PETR4|quarterly|2025-09-30|127906000000| -66789000000|61117000000|               NULL|                 -2729000000|    

In [11]:
df = spark.table("valueaddedhistoryquarterly")
df.select(*[column for column in df.columns][:10]).show(5)

+-----+---------+----------+------------+------------+-------------+-----------------------+-------------------------------------+---------------------------------+---------------------+
|stock|     type|   endDate|     revenue|productSales|otherRevenues|constructionOfOwnAssets|provisionOrReversalOfDoubtfulAccounts|suppliesPurchasedFromThirdParties|costsWithProductsSold|
+-----+---------+----------+------------+------------+-------------+-----------------------+-------------------------------------+---------------------------------+---------------------+
|PETR4|quarterly|2026-03-31|191003000000|159497000000|   7693000000|            23770000000|                             43000000|                     -66740000000|         -20962000000|
|PETR4|quarterly|2025-12-31|194300000000|164846000000|   4038000100|            25642000000|                           -226000000|                     -83922000000|         -25447000000|
|PETR4|quarterly|2025-09-30|197877000000|165526000000|   46390000